In [75]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

In [76]:
start, end = '2022-01-01', '2026-01-01'

close_prices = yf.download('SPY', start=start, end=end, interval='1d', auto_adjust=True)['Close']

print(f'Shape: {close_prices.shape}')
display(close_prices.head())
display(close_prices.describe())
display(close_prices.info())

# convert dataframe to series (use SPY for now)
close_prices = close_prices['SPY']

[*********************100%***********************]  1 of 1 completed

Shape: (1003, 1)


Ticker,SPY
Date,
2022-01-03,450.644501
2022-01-04,450.493500
2022-01-05,441.842987
2022-01-06,441.427948
2022-01-07,439.682770


Ticker,SPY
count,1003.000000
mean,486.070600
std,97.814341
min,340.252869
25%,397.364975
50%,461.188263
75%,569.474579
max,688.499695


<class 'pandas.DataFrame'>
DatetimeIndex: 1003 entries, 2022-01-03 to 2025-12-31
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   SPY     1003 non-null   float64
dtypes: float64(1)
memory usage: 15.7 KB


None

In [77]:
# derive 3 features (returns, volatility, drawdowns)

vol_window = 5          # 5-10 days is reactive and quick capturing day-to-day turbulence
drawdown_window = 60    # 60 days ~1 quarter and is cumulative capturing how far below meaningful recent high

log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
rolling_vol = close_prices.rolling(window=vol_window).std()
rolling_max = close_prices.rolling(window=drawdown_window).max()
drawdowns = (close_prices - rolling_max) / close_prices

features = pd.DataFrame({
    'log_return': log_returns,
    'rolling_vol': rolling_vol,
    'drawdown': drawdowns,
}).dropna()

print(f'Shape: {features.shape}')
display(features.head())
display(features.describe())
display(features.info())


Shape: (944, 3)


,log_return,rolling_vol,drawdown
Date,,,
2022-03-29,0.012295,6.214962,-0.031807
2022-03-30,-0.006194,4.214108,-0.037870
2022-03-31,-0.015511,3.902108,-0.039680
2022-04-01,0.002830,3.858110,-0.036742
2022-04-04,0.008530,3.864405,-0.027936


,log_return,rolling_vol,drawdown
count,944.000000,944.000000,944.000000
mean,0.000482,4.236316,-0.037573
std,0.011166,2.842897,0.051087
min,-0.060327,0.318043,-0.258830
25%,-0.004548,2.528639,-0.051500
50%,0.000671,3.562857,-0.014213
75%,0.006265,5.320340,-0.002564
max,0.099863,28.686828,0.000000


<class 'pandas.DataFrame'>
DatetimeIndex: 944 entries, 2022-03-29 to 2025-12-31
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   log_return   944 non-null    float64
 1   rolling_vol  944 non-null    float64
 2   drawdown     944 non-null    float64
dtypes: float64(3)
memory usage: 29.5 KB


None

In [ ]:
# include specific input features
feature_cols = ['log_return', 'rolling_vol', 'drawdown']

# standardize feature data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features[feature_cols])

# fit hmm with 3 hidden states
model = GaussianHMM(n_components=3, covariance_type='full', n_iter=1000, random_state=42)
model.fit(X_scaled)

print(f'Converged: {model.monitor_.converged}')
print(f'Log-likelihood: {model.score(X_scaled):.2f}')

# predict regimess
features['regime'] = model.predict(X_scaled)

print(f'\nRegime counts:\n{features['regime'].value_counts().sort_index()}')

display(features)

Converged: True
Log-likelihood: -1943.97

Regime counts:
regime
0    403
1    205
2     25
3    311
Name: count, dtype: int64


,log_return,rolling_vol,drawdown,regime
Date,,,,
2022-03-29,0.012295,6.214962,-0.031807,1
2022-03-30,-0.006194,4.214108,-0.037870,3
2022-03-31,-0.015511,3.902108,-0.039680,3
2022-04-01,0.002830,3.858110,-0.036742,3
2022-04-04,0.008530,3.864405,-0.027936,3
...,...,...,...,...
2025-12-24,0.003511,6.283038,0.000000,3
2025-12-26,-0.000101,4.139765,-0.000101,0
2025-12-29,-0.003570,2.269732,-0.003678,0
